# The Full Campaign Lifecycle: Submission to Calendar

This notebook covers the v2.2 milestone (Phases 26-29) end to end -- a campaign
(`tom_targets.TargetList`) and four `CampaignRun`s, taken all the way from public
submission to calendar events -- by driving every state transition through the REAL
staff-facing views with `django.test.Client`. Every write below goes through the same
`campaigns:submit`, `campaigns:decide` (`approve`/`resolve_site`) and
`campaigns:attribution_decide` (`confirm`) views a real submitter and a real staff member
would use.

This deliberately contrasts with `reconcile_campaign_runs_demo.ipynb`, which covers the
reconciler alone (`solsys_code/campaign_reconciler.py`), seeding its four `CampaignRun`s
directly against already-approved rows via `CampaignRun.objects.update_or_create(...,
approval_status=APPROVED)`. That is the right scope for demonstrating the reconciler in
isolation, but it skips every human workflow step the milestone actually added -- the
public intake form, the approval decision, site disambiguation, and operator-assisted
attribution. This notebook is the missing counterpart: it demonstrates those human
workflow steps, and only reaches the reconciler as their consequence.

This notebook lives in `pre_executed/` because it is **DB-dependent** (it seeds
`Observatory`/`TargetList` records and creates `CampaignRun`/`CalendarEvent` rows) and is
therefore **NOT** run during Sphinx/CI/ReadTheDocs builds, per `docs/notebooks/README.md`.

## Django setup, and why this notebook calls `setup_test_environment()`

Standard boilerplate to make `src.fomo.settings` importable from this notebook's location
(`docs/notebooks/pre_executed/` -- three levels under the repo root, so `parents[2]` gives
the repo root) and to allow synchronous ORM calls inside Jupyter's async event loop, exactly
like `reconcile_campaign_runs_demo.ipynb`'s own setup cell.

This notebook additionally calls `django.test.utils.setup_test_environment()` once, guarded
in a `try`/`except RuntimeError` so a re-run of this notebook in the same kernel session
does not fail on the already-called guard. That call does two things, both required below:

1. It appends `testserver` -- `django.test.Client`'s default `SERVER_NAME` -- to
   `ALLOWED_HOSTS`. Without it, a bare `Client()` request raises `DisallowedHost`: this
   developer's `src/fomo/local_settings.py` (gitignored) sets a non-empty `ALLOWED_HOSTS`,
   and a developer without that file gets `ALLOWED_HOSTS = []` with `DEBUG = True`, which
   permits only localhost -- `testserver` is neither.
2. It swaps in Django's locmem email backend, so `CampaignRunSubmissionView._notify_staff()`'s
   staff-notification email is captured in `django.core.mail.outbox` below instead of being
   dumped to this notebook's console output.

**There is no separate test database here.** `django.test.Client` drives the real
request/response cycle straight through the ordinary views against this developer's own
dev database (`src/fomo_db.sqlite3`) -- the same database the running site itself uses.
That is exactly why every write below is demo-scoped: the reset cell further down deletes
only this demo campaign's own rows, by exact name, before creating anything.

In [1]:
import os
import sys
from pathlib import Path

import django
import django.test.utils

# Ensure the repo root is on sys.path so `src.fomo.settings` is importable
# when this notebook is executed from docs/notebooks/pre_executed/.
# NOTE: parents[2] is correct only when the Jupyter kernel CWD is
# docs/notebooks/pre_executed/. Start Jupyter from that directory, or
# adjust the index if you launch from the repo root.
repo_root_path = Path.cwd().resolve().parents[2]
if not (repo_root_path / 'manage.py').exists():
    raise RuntimeError(f'No manage.py at {repo_root_path}; run Jupyter from docs/notebooks/pre_executed/')
repo_root = str(repo_root_path)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'src.fomo.settings')

# Jupyter's ipykernel runs inside an asyncio event loop, but Django's ORM is
# sync-only by default and refuses to run there; this opts back in.
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

django.setup()

# This notebook intentionally imports only the campaign-coordination models/views and the
# reconciler below -- never the ephemeris view/computation modules, which trigger a large
# one-time SPICE kernel download on first import.

try:
    django.test.utils.setup_test_environment()
except RuntimeError:
    # Already set up by an earlier run of this notebook in the same kernel session.
    pass

print(f'Django ready: settings module={os.environ["DJANGO_SETTINGS_MODULE"]!r}, repo_root={repo_root!r}')

Django ready: settings module='src.fomo.settings', repo_root='/home/tlister/git/fomo_devel/.claude/worktrees/agent-a6c22df11e68ab4db'


## Staff user and two test clients

Find-or-create one demo staff `User` (`is_staff=True`) with `get_or_create()` +
`set_unusable_password()`, so no real credential is ever added to the dev database. Two
`django.test.Client` instances are built from it:

- `public_client` -- never logged in. Used for the public submission form and the public
  campaign table.
- `staff_client` -- `force_login(staff_user)`. Used for the approval queue and the
  attribution queue.

Two clients, not one: `campaigns:approval_queue`, `campaigns:decide`,
`campaigns:attribution` and `campaigns:attribution_decide` are all `StaffRequiredMixin`-
gated -- using a genuinely anonymous client for the public paths (`campaigns:submit`,
`campaigns:table`) proves those paths really need no login, rather than merely asserting it.

In [2]:
from django.contrib.auth.models import User
from django.test import Client

DEMO_STAFF_USERNAME = 'campaign_lifecycle_demo_staff'

staff_user, staff_created = User.objects.get_or_create(
    username=DEMO_STAFF_USERNAME,
    defaults=dict(email='campaign-lifecycle-demo-staff@example.org', is_staff=True),
)
if staff_created:
    staff_user.set_unusable_password()
    staff_user.save()
print(
    f'Staff user: {staff_user.username!r} (pk={staff_user.pk}) '
    f'{"created" if staff_created else "found"}, is_staff={staff_user.is_staff}'
)

public_client = Client()
staff_client = Client()
staff_client.force_login(staff_user)
print('public_client: anonymous (never logged in)')
print(f'staff_client: force_login({staff_user.username!r})')

User campaign_lifecycle_demo_staff logged in without a password. Cannot create encryption key.


Staff user: 'campaign_lifecycle_demo_staff' (pk=2) found, is_staff=True
public_client: anonymous (never logged in)
staff_client: force_login('campaign_lifecycle_demo_staff')


## Demo-scoped reset

Makes the whole notebook re-runnable against any dev DB. This cell deletes ONLY this
demo's own rows: every `CampaignRun` whose `campaign` is the `Campaign Lifecycle Demo`
`TargetList`, every `CalendarEvent` whose `target_list` is that same `TargetList`, and any
`CalendarEvent` whose `url` starts with this demo's own legacy-event prefix (the
hand-entered event created further down, to demonstrate the attribution queue). It is
guarded on that `TargetList` already existing -- on a first-ever run there is nothing to
delete.

This is a **real dev database** shared with the 3I/ATLAS campaigns and everything else in
it, so the reset is scoped by campaign name and never touches anything wider than the demo
campaign -- no blanket `CampaignRun.objects.all().delete()`, ever. This is required because
the public submission path below uses `CampaignRun.objects.create()`, which is not
idempotent and would otherwise trip one of `CampaignRun`'s two natural-key
`UniqueConstraint`s on a second execution of this notebook.

In [3]:
from tom_calendar.models import CalendarEvent
from tom_targets.models import TargetList

from solsys_code.models import CampaignRun

DEMO_CAMPAIGN_NAME = 'Campaign Lifecycle Demo'
# Namespaced distinctly from the reconciler demo's own 'RUN:' key family, and from that
# notebook's obscodes X29/X30, so the two demo notebooks never interfere with each other.
DEMO_LEGACY_EVENT_URL_PREFIX = 'LEGACY:campaign-lifecycle-demo:'

existing_demo_campaign = TargetList.objects.filter(name=DEMO_CAMPAIGN_NAME).first()
if existing_demo_campaign is None:
    print(f'No prior {DEMO_CAMPAIGN_NAME!r} campaign found -- nothing to reset (first run).')
else:
    deleted_runs, _ = CampaignRun.objects.filter(campaign=existing_demo_campaign).delete()
    deleted_campaign_events, _ = CalendarEvent.objects.filter(target_list=existing_demo_campaign).delete()
    deleted_legacy_events, _ = CalendarEvent.objects.filter(url__startswith=DEMO_LEGACY_EVENT_URL_PREFIX).delete()
    print(
        f'Reset {DEMO_CAMPAIGN_NAME!r}: deleted {deleted_runs} CampaignRun row(s) (cascade-deletes '
        f'their own RUN:-namespaced events), {deleted_campaign_events} additional CalendarEvent '
        f'row(s) still linked to the campaign, {deleted_legacy_events} legacy-prefixed CalendarEvent '
        f'row(s).'
    )

Reset 'Campaign Lifecycle Demo': deleted 4 CampaignRun row(s) (cascade-deletes their own RUN:-namespaced events), 2 additional CalendarEvent row(s) still linked to the campaign, 0 legacy-prefixed CalendarEvent row(s).


## Seed Observatory records and the campaign TargetList

Three ground `Observatory` rows, `update_or_create`d so this cell is safe to re-run, each
with a real IANA `timezone` so `sun_event()` can compute dip-corrected sunset/sunrise:

- `Y21` -- a Chilean-style classical site (`America/Santiago`).
- `Y22` -- an LCO-network-style site (`Australia/Sydney`, an LCO network node location).
- `Y23` -- a Paranal-shaped site for the ESO run, modelled on the real VLT/FORS2 site at MPC
  309 Paranal in name, coordinates and altitude, but kept under this demo's own obscode
  rather than writing to the real `309` row.

`X29`/`X30` are `reconcile_campaign_runs_demo.ipynb`'s own obscodes -- deliberately not
reused here, so the two demo notebooks never interfere with each other.

The campaign container is a `tom_targets.models.TargetList`, found-or-created by name.

In [4]:
from tom_targets.models import TargetList

from solsys_code.solsys_code_observatory.models import Observatory

y21_site, _ = Observatory.objects.update_or_create(
    obscode='Y21',
    defaults=dict(
        name='Y21 Classical Site (Campaign Lifecycle Demo)',
        short_name='Y21 Classical',
        lat=-30.1697,
        lon=-70.8065,
        altitude=2207,
        timezone='America/Santiago',
        observations_type=Observatory.OPTICAL_OBSTYPE,
    ),
)
print(f'Y21 (classical): obscode={y21_site.obscode!r}  timezone={y21_site.timezone!r}')

y22_site, _ = Observatory.objects.update_or_create(
    obscode='Y22',
    defaults=dict(
        name='Y22 LCO Network Site (Campaign Lifecycle Demo)',
        short_name='Y22 LCO Siding Spring',
        lat=-31.2733,
        lon=149.0644,
        altitude=1116,
        timezone='Australia/Sydney',
        observations_type=Observatory.OPTICAL_OBSTYPE,
    ),
)
print(f'Y22 (LCO queue): obscode={y22_site.obscode!r}  timezone={y22_site.timezone!r}')

y23_site, _ = Observatory.objects.update_or_create(
    obscode='Y23',
    defaults=dict(
        name='Y23 Paranal-shaped Site (Campaign Lifecycle Demo, modeled on real VLT/FORS2 at MPC 309)',
        short_name='Y23 VLT/FORS2-shaped',
        lat=-24.6275,
        lon=-70.4044,
        altitude=2635,
        timezone='America/Santiago',
        observations_type=Observatory.OPTICAL_OBSTYPE,
    ),
)
print(
    f'Y23 (ESO queue, resolved later via Sites Needing Review): '
    f'obscode={y23_site.obscode!r}  timezone={y23_site.timezone!r}'
)

campaign, campaign_created = TargetList.objects.get_or_create(name=DEMO_CAMPAIGN_NAME)
print(f'\nCampaign: {campaign.name!r} (pk={campaign.pk}) {"created" if campaign_created else "found"}')

Y21 (classical): obscode='Y21'  timezone='America/Santiago'
Y22 (LCO queue): obscode='Y22'  timezone='Australia/Sydney'
Y23 (ESO queue, resolved later via Sites Needing Review): obscode='Y23'  timezone='America/Santiago'

Campaign: 'Campaign Lifecycle Demo' (pk=1) found


## Four submissions through the public form

One `public_client.post()` to `campaigns:submit` per run, using the SAME anonymous client
that never logs in -- the public submission path genuinely needs no authentication. Each
POST asserts a 302 redirect to `campaigns:submission_thanks`, then looks the created run
back up by its natural key (`campaign` + `telescope_instrument`) rather than assuming a pk,
since a re-run of this notebook against a shared dev DB may not always land on the same pk
sequence.

The four submissions:

- **Classical** -- `site_raw='Y21'`, a 3-night range.
- **LCO queue** -- a distinct `telescope_instrument`, `site_raw='Y22'`, a 3-night range.
- **ESO queue** -- a VLT/FORS2-shaped `telescope_instrument` (`'UT1/FORS2'`), and a
  `site_raw` that is deliberately unresolvable free text longer than 4 characters
  (`'Paranal Observatory (site TBC)'`) so site resolution fails at approve time and the run
  lands in Sites Needing Review -- resolved later, in Task 2, via `Y23`.
- **Class-wide** -- a site-agnostic `telescope_instrument`, `site_raw` left blank, and a
  whole-month window.

Every submission supplies the required `contact_person`/`contact_email` and leaves the
`alt_contact_info` honeypot blank (by omitting it -- the form's `clean_alt_contact_info()`
treats a missing value the same as an explicit blank string). The printed output below
shows all four runs arriving as `source='web'` / `approval_status='pending_review'` with no
site and no calendar event yet, plus the growing `django.core.mail.outbox` -- the staff
notifications the view sent.

In [5]:
from django.core import mail
from django.urls import reverse

from solsys_code.models import CampaignRun

CONTACT_PERSON = 'Dr. Lifecycle Demo'
CONTACT_EMAIL = 'lifecycle-demo-contact@example.org'

SUBMISSIONS = [
    (
        'Classical',
        dict(
            campaign=campaign.pk,
            telescope_instrument='Y21 0.4m/SBIG-STX16803',
            site_raw='Y21',
            obs_date='2026-09-01 to 2026-09-03',
            contact_person=CONTACT_PERSON,
            contact_email=CONTACT_EMAIL,
        ),
    ),
    (
        'LCO queue',
        dict(
            campaign=campaign.pk,
            telescope_instrument='Y22 1m0-SciCam-Sinistro',
            site_raw='Y22',
            obs_date='2026-09-05 to 2026-09-07',
            contact_person=CONTACT_PERSON,
            contact_email=CONTACT_EMAIL,
        ),
    ),
    (
        'ESO queue',
        dict(
            campaign=campaign.pk,
            telescope_instrument='UT1/FORS2',
            site_raw='Paranal Observatory (site TBC)',
            obs_date='2026-09-10 to 2026-09-12',
            contact_person=CONTACT_PERSON,
            contact_email=CONTACT_EMAIL,
        ),
    ),
    (
        'Class-wide',
        dict(
            campaign=campaign.pk,
            telescope_instrument='LCO 1m0 Network (Campaign Lifecycle Demo)',
            site_raw='',
            obs_date='2026-09-01 to 2026-09-30',
            contact_person=CONTACT_PERSON,
            contact_email=CONTACT_EMAIL,
        ),
    ),
]

submitted_runs = {}
for label, form_data in SUBMISSIONS:
    response = public_client.post(reverse('campaigns:submit'), data=form_data)
    assert (
        response.status_code == 302
    ), f'{label} submission failed: {response.status_code} {getattr(response, "context", None)}'
    assert response.url == reverse('campaigns:submission_thanks')
    run = CampaignRun.objects.get(campaign=campaign, telescope_instrument=form_data['telescope_instrument'])
    submitted_runs[label] = run
    print(
        f'{label:<12} pk={run.pk}  source={run.source!r}  approval_status={run.approval_status!r}  '
        f'site_raw={run.site_raw!r}  window={run.window_start}..{run.window_end}  '
        f'mail.outbox so far: {len(mail.outbox)}'
    )

classical_run = submitted_runs['Classical']
lco_queue_run = submitted_runs['LCO queue']
eso_queue_run = submitted_runs['ESO queue']
class_wide_run = submitted_runs['Class-wide']

Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.


NumExpr defaulting to 16 threads.


Using fallback library next to module: /home/tlister/venv/devel_fomo311_venv/lib/python3.11/site-packages/spiceypy/utils/libcspice.so


registering new views: args: ('groups', <class 'tom_common.api_views.GroupViewSet'>, 'groups'), kwargs: {}


registering new views: args: ('targets', <class 'tom_targets.api_views.TargetViewSet'>, 'targets'), kwargs: {}


...

registering new views: args: ('targetlist', <class 'tom_targets.api_views.TargetListViewSet'>, 'targetlist'), kwargs: {}


registering new views: args: ('observations', <class 'tom_observations.api_views.ObservationRecordViewSet'>, 'observations'), kwargs: {}


registering new views: args: ('dataproducts', <class 'tom_dataproducts.api_views.DataProductViewSet'>, 'dataproducts'), kwargs: {}


registering new views: args: ('reduceddatums', <class 'tom_dataproducts.api_views.ReducedDatumViewSet'>, 'reduceddatums'), kwargs: {}


Classical    pk=25  source='web'  approval_status='pending_review'  site_raw='Y21'  window=2026-09-01..2026-09-03  mail.outbox so far: 1
LCO queue    pk=26  source='web'  approval_status='pending_review'  site_raw='Y22'  window=2026-09-05..2026-09-07  mail.outbox so far: 2
ESO queue    pk=27  source='web'  approval_status='pending_review'  site_raw='Paranal Observatory (site TBC)'  window=2026-09-10..2026-09-12  mail.outbox so far: 3
Class-wide   pk=28  source='web'  approval_status='pending_review'  site_raw=''  window=2026-09-01..2026-09-30  mail.outbox so far: 4


## Provenance stamping, with the honesty note

The public submission form exposes neither `source` nor `telescope_class` -- in production
those values come from the ingest path that CREATED the row. `import_campaign_csv` already
writes `source=CSV_IMPORT`; v2.3's ADAPT-01..03 will rewire the three calendar-sync
adapters to write `LCO_QUEUE`/`GEMINI_QUEUE`/`ESO_QUEUE`. Those adapter paths do not exist
yet, so this notebook stamps the four values by hand -- purely to obtain four rows of the
right provenance for the walkthrough below. **This is not a shortcut for the workflow
itself**: every *state transition* below (submission, approval, site resolution,
attribution) still goes through a real staff-facing view (D-01); only these two fields,
which the public form was never given a way to set, are written directly here.

`telescope_class` must be set on the class-wide run BEFORE it is approved in Task 2 below,
because `CampaignRunDecisionView.post()`'s approve branch computes `site_needs_review =
needs_review and not run.telescope_class`, and `reconcile_run()` dispatches on
`telescope_class` first.

In [6]:
classical_run.source = CampaignRun.Source.CLASSICAL_FILE
classical_run.save(update_fields=['source'])

lco_queue_run.source = CampaignRun.Source.LCO_QUEUE
lco_queue_run.save(update_fields=['source'])

eso_queue_run.source = CampaignRun.Source.ESO_QUEUE
eso_queue_run.save(update_fields=['source'])

class_wide_run.source = CampaignRun.Source.LEGACY
class_wide_run.telescope_class = CampaignRun.TelescopeClass.ONE_M0
class_wide_run.save(update_fields=['source', 'telescope_class'])

for label, run in submitted_runs.items():
    run.refresh_from_db()
    print(f'{label:<12} pk={run.pk}  source={run.source!r}  telescope_class={run.telescope_class!r}')

Classical    pk=25  source='classical_file'  telescope_class=''
LCO queue    pk=26  source='lco_queue'  telescope_class=''
ESO queue    pk=27  source='eso_queue'  telescope_class=''
Class-wide   pk=28  source='legacy'  telescope_class='1m0'


## The approval queue, before

`staff_client.get(reverse('campaigns:approval_queue'))` -- gated by `StaffRequiredMixin`,
so only a logged-in staff user can reach it. The view builds its site-candidate pool
(`build_site_candidates()`) once per request; that pool is 24h-cached and falls back to a
local-only pool on any MPC API failure, so a cold cache may make one outbound MPC call here
but never fails. Sites Needing Review renders first on this page (plan 27-07) -- empty for
now, since none of the four demo runs has been approved yet.

In [7]:
response = staff_client.get(reverse('campaigns:approval_queue'))
assert response.status_code == 200

pending_pks = {run.pk for run in response.context['pending_table'].data}
review_pks = {run.pk for run in response.context['review_table'].data}
print(f'Pending review (pending_table): {sorted(pending_pks)}')
print(f'Needing site review (review_table), before any approval: {sorted(review_pks)}')

demo_pks = {run.pk for run in submitted_runs.values()}
assert demo_pks <= pending_pks, 'All four demo runs should still be pending review'

Pending review (pending_table): [25, 26, 27, 28]
Needing site review (review_table), before any approval: []


## Approve all four

One `staff_client.post(reverse('campaigns:decide', args=[run.pk]), {'action': 'approve'})`
per run -- no `site_selection` in the POST body, so `_resolve_site` falls back to each
run's own `site_raw`. The executed output below makes three facts visible:

- The `Y21`/`Y22` runs resolve their sites via tier 1 (an exact local `Observatory.obscode`
  match, no network) and immediately get their per-night events, because `approve()` calls
  `reconcile_run()` itself.
- The ESO run is approved anyway, with `site=None` and `site_needs_review=True` -- site
  resolution failure never blocks approval -- and therefore has no events yet.
- The class-wide run gets its container event with `site` still `None` and
  `site_needs_review` still `False`, because its `telescope_class` (set in Task 1) already
  answers "why is there no site" (models.py D-06) -- it was never eligible for the Sites
  Needing Review queue in the first place.

In [8]:
from solsys_code.campaign_reconciler import owned_events

for label, run in submitted_runs.items():
    response = staff_client.post(reverse('campaigns:decide', args=[run.pk]), {'action': 'approve'})
    assert response.status_code == 302
    assert response.url == reverse('campaigns:approval_queue')
    run.refresh_from_db()
    print(
        f'{label:<12} pk={run.pk}  approval_status={run.approval_status!r}  site={run.site!r}  '
        f'site_needs_review={run.site_needs_review}  events={owned_events(run).count()}'
    )

assert classical_run.site is not None and not classical_run.site_needs_review
assert lco_queue_run.site is not None and not lco_queue_run.site_needs_review
assert eso_queue_run.site is None and eso_queue_run.site_needs_review
assert class_wide_run.site is None and not class_wide_run.site_needs_review

Classical    pk=25  approval_status='approved'  site=<Observatory: Y21: Y21 Classical Site (Campaign Lifecycle Demo)>  site_needs_review=False  events=3


LCO queue    pk=26  approval_status='approved'  site=<Observatory: Y22: Y22 LCO Network Site (Campaign Lifecycle Demo)>  site_needs_review=False  events=3
ESO queue    pk=27  approval_status='approved'  site=None  site_needs_review=True  events=0
Class-wide   pk=28  approval_status='approved'  site=None  site_needs_review=False  events=1


## Site-review resolution

GET the approval queue again: the ESO run now appears in Sites Needing Review. Then
`staff_client.post(reverse('campaigns:decide', args=[eso_queue_run.pk]), {'action':
'resolve_site', 'site_selection': 'Y23'})` resolves it against the Paranal-shaped `Y23`
site seeded in Task 1. `_resolve_site()` clears `site_needs_review` only AFTER
`reconcile_run()` succeeds -- so a failed projection would leave the row in the queue for a
retry rather than silently losing it.

In [9]:
response = staff_client.get(reverse('campaigns:approval_queue'))
assert response.status_code == 200
review_pks_after_approve = {run.pk for run in response.context['review_table'].data}
print(f'Needing site review, after approval: {sorted(review_pks_after_approve)}')
assert eso_queue_run.pk in review_pks_after_approve

response = staff_client.post(
    reverse('campaigns:decide', args=[eso_queue_run.pk]),
    {'action': 'resolve_site', 'site_selection': 'Y23'},
)
assert response.status_code == 302
eso_queue_run.refresh_from_db()
print(
    f'ESO queue run pk={eso_queue_run.pk}  site={eso_queue_run.site!r}  '
    f'site_needs_review={eso_queue_run.site_needs_review}  events={owned_events(eso_queue_run).count()}'
)
assert eso_queue_run.site == y23_site
assert not eso_queue_run.site_needs_review
assert owned_events(eso_queue_run).count() > 0

Needing site review, after approval: [27]


ESO queue run pk=27  site=<Observatory: Y23: Y23 Paranal-shaped Site (Campaign Lifecycle Demo, modeled on real VLT/FORS2 at MPC 309)>  site_needs_review=False  events=3


## An orphan calendar event, and the attribution queue

A `load_telescope_runs`-style entry that predates the canonical run record is represented
here by creating one `CalendarEvent` directly -- a hand-entered/legacy-sync-created row
with NO `CalendarEventMeta` companion row at all, which is precisely the orphan kind
`orphan_calendar_events()`'s first branch exists to catch. Its `target_list` is set to the
demo campaign (required -- `_eligible_runs_for_event()` hard-gates on it), its
`telescope`/`instrument` match the classical run's own two halves, and its window overlaps
the classical run's window. Its `url` uses this demo's own legacy prefix, which the Task 1
reset cell already cleans up.

`staff_client.get(reverse('campaigns:attribution'))` confirms it is now in the backlog.
Its scored candidate list is then printed by calling `campaign_attribution.
candidates_for_event()` directly -- the same function the view's context uses -- rather
than depending on which pagination page it happens to land on among any other pre-existing
orphans already in this dev DB. Three scoring signals: date overlap (weight 0.40),
instrument similarity (0.35) and telescope match (0.25). Bands are High (>=0.75), Medium
(>=0.50), else Low. The hard campaign boundary: only runs in the SAME campaign as the event
are ever scored at all (`_eligible_runs_for_event()`).

In [10]:
from datetime import datetime
from datetime import timezone as dt_timezone

from tom_calendar.models import CalendarEvent

from solsys_code import campaign_attribution

classical_telescope, _sep, classical_instrument = classical_run.telescope_instrument.partition('/')

orphan_event = CalendarEvent.objects.create(
    title=f'{campaign.name}: hand-entered pre-canon night (demo)',
    description='A pre-existing, hand-entered calendar entry that predates the canonical CampaignRun record.',
    start_time=datetime(2026, 9, 1, 22, 0, tzinfo=dt_timezone.utc),
    end_time=datetime(2026, 9, 2, 10, 0, tzinfo=dt_timezone.utc),
    url=f'{DEMO_LEGACY_EVENT_URL_PREFIX}2026-09-01',
    target_list=campaign,
    telescope=classical_telescope,
    instrument=classical_instrument,
)
print(
    f'Orphan event created: pk={orphan_event.pk}  url={orphan_event.url!r}  '
    f'telescope={orphan_event.telescope!r}  instrument={orphan_event.instrument!r}'
)
assert not hasattr(orphan_event, 'telescope_label_meta'), 'orphan event must have no CalendarEventMeta yet'

response = staff_client.get(reverse('campaigns:attribution'))
assert response.status_code == 200
print(f'attribution_count={response.context["attribution_count"]}  is_drained={response.context["is_drained"]}')

candidates = campaign_attribution.candidates_for_event(orphan_event)
print(f'Orphan event pk={orphan_event.pk} candidates:')
for candidate in candidates:
    print(
        f'  run pk={candidate.run.pk}  source={candidate.run.source!r}  '
        f'score={candidate.score:.2f}  band={candidate.band}'
    )
assert any(c.run.pk == classical_run.pk for c in candidates), 'classical_run should be a scored candidate'

Orphan event created: pk=55  url='LEGACY:campaign-lifecycle-demo:2026-09-01'  telescope='Y21 0.4m'  instrument='SBIG-STX16803'
attribution_count=1  is_drained=False
Orphan event pk=55 candidates:
  run pk=25  source='classical_file'  score=0.82  band=high
  run pk=28  source='legacy'  score=0.58  band=medium
  run pk=26  source='lco_queue'  score=0.21  band=low
  run pk=27  source='eso_queue'  score=0.20  band=low


## Confirm the attribution

`staff_client.post(reverse('campaigns:attribution_decide'), {'action': 'confirm', 'kind':
'event', 'orphan_pk': orphan_event.pk, 'run_pk': classical_run.pk})` writes the
confirmation. The view re-derives eligibility server-side via `campaign_attribution.
is_offered_candidate()` before writing -- a tampered POST cannot create an association
across a campaign boundary. This attribution also survives future reconciler sweeps:
`_detach_stale_family_events()` is scoped to the classical run's own `RUN:{pk}` url
namespace, and this orphan event's url is outside it entirely.

In [11]:
from solsys_code.models import CalendarEventMeta

response = staff_client.post(
    reverse('campaigns:attribution_decide'),
    {'action': 'confirm', 'kind': 'event', 'orphan_pk': orphan_event.pk, 'run_pk': classical_run.pk},
)
assert response.status_code == 302

meta = CalendarEventMeta.objects.get(event_id=orphan_event.pk)
print(
    f'CalendarEventMeta for event pk={orphan_event.pk}: run={meta.run!r}  '
    f'confirmed_by={meta.confirmed_by!r}  confirmed_at={meta.confirmed_at!r}'
)
assert meta.run_id == classical_run.pk
assert meta.confirmed_by_id == staff_user.pk
assert meta.confirmed_at is not None

response = staff_client.get(reverse('campaigns:attribution'))
assert response.status_code == 200
print(f'attribution_count after confirming: {response.context["attribution_count"]}')

CalendarEventMeta for event pk=55: run=<CampaignRun: #25 Campaign Lifecycle Demo | Y21 0.4m/SBIG-STX16803 | 2026-09-01..2026-09-03 | Y21>  confirmed_by=<User: campaign_lifecycle_demo_staff>  confirmed_at=datetime.datetime(2026, 8, 7, 1, 20, 42, 386657, tzinfo=datetime.timezone.utc)
attribution_count after confirming: 0


## The four-way payoff

This is the cell the whole notebook exists for. Looping over all four runs, in submission
order, and printing every `campaign_reconciler.owned_events(run)` row (the same
ownership-scoped query the reconciler itself uses) ordered by `start_time` -- the executed
output below makes the shape difference plainly visible.

`reconcile_run()` reads `CampaignRun.source` NOWHERE in its dispatch. It branches on a
non-blank `telescope_class` first, then on a satellite `site`, and everything else --
classical or queue-scheduled alike -- takes the per-night branch. Quick task `260805-tad`
removed the last source-based branch that used to exist here. So three of these four runs
below render identically (one date-bearing `RUN:{pk}:{date}` event per observing night) --
the classical run, the LCO-queue run and the ESO-queue run -- despite carrying three
different `source` values (`classical_file`, `lco_queue`, `eso_queue`), and only the
class-wide run (`telescope_class` set) renders a single bare `RUN:{pk}` container spanning
its whole window. `source` remains worth setting correctly, as the run's provenance record
that staff views and reports read -- it just does not change an event's shape. See "How do
I get every campaign run onto the calendar?" in `docs/runbooks/telescope_runs_calendar.rst`
for the operator-facing version of this note.

In [12]:
import re

RUN_NIGHT_RE = re.compile(r'^RUN:(\d+):\d{4}-\d{2}-\d{2}$')

per_night_run_pks = set()
container_only_run_pks = set()

for label, run in [
    ('Classical', classical_run),
    ('LCO queue', lco_queue_run),
    ('ESO queue', eso_queue_run),
    ('Class-wide', class_wide_run),
]:
    print(
        f'--- {label} run (pk={run.pk}, source={run.source!r}, '
        f'telescope_class={run.telescope_class!r}, site={run.site!r}) ---'
    )
    events = list(owned_events(run).order_by('start_time'))
    night_urls = [ev.url for ev in events if RUN_NIGHT_RE.match(ev.url)]
    if night_urls:
        per_night_run_pks.add(run.pk)
    else:
        container_only_run_pks.add(run.pk)
    for ev in events:
        print(f'  url={ev.url!r}')
        print(f'    title={ev.title!r}')
        print(f'    telescope={ev.telescope!r}  instrument={ev.instrument!r}')
        print(f'    start={ev.start_time.isoformat()}  end={ev.end_time.isoformat()}')
    print()

print(f'Per-night runs (RUN:{{pk}}:{{date}}): {sorted(per_night_run_pks)}')
print(f'Container-only runs (bare RUN:{{pk}}): {sorted(container_only_run_pks)}')

assert per_night_run_pks == {classical_run.pk, lco_queue_run.pk, eso_queue_run.pk}, (
    'Classical, LCO-queue and ESO-queue runs must all render per-night events -- source does '
    'not select the branch, only telescope_class/site do.'
)
assert container_only_run_pks == {
    class_wide_run.pk
}, 'Only the class-wide run (telescope_class set) should render the bare whole-window container.'

--- Classical run (pk=25, source='classical_file', telescope_class='', site=<Observatory: Y21: Y21 Classical Site (Campaign Lifecycle Demo)>) ---
  url='RUN:25:2026-09-01'
    title='Campaign Lifecycle Demo: Y21 0.4m/SBIG-STX16803 (window 2026-09-01..2026-09-03)'
    telescope='Y21 0.4m'  instrument='SBIG-STX16803'
    start=2026-09-01T22:34:40+00:00  end=2026-09-02T10:50:54+00:00
  url='RUN:25:2026-09-02'
    title='Campaign Lifecycle Demo: Y21 0.4m/SBIG-STX16803 (window 2026-09-01..2026-09-03)'
    telescope='Y21 0.4m'  instrument='SBIG-STX16803'
    start=2026-09-02T22:35:12+00:00  end=2026-09-03T10:49:43+00:00
  url='RUN:25:2026-09-03'
    title='Campaign Lifecycle Demo: Y21 0.4m/SBIG-STX16803 (window 2026-09-01..2026-09-03)'
    telescope='Y21 0.4m'  instrument='SBIG-STX16803'
    start=2026-09-03T22:35:44+00:00  end=2026-09-04T10:48:32+00:00

--- LCO queue run (pk=26, source='lco_queue', telescope_class='', site=<Observatory: Y22: Y22 LCO Network Site (Campaign Lifecycle Demo)>) 

## The public campaign table

`public_client.get(reverse('campaigns:table', args=[campaign.pk]))` -- the SAME anonymous
client used for the submission form, never logged in. This is the public read path
(VIEW-01/04): soft-filtered via `.values()` so pending rows are excluded from the SQL
SELECT entirely (in contrast to the two staff queues above, both hard-gated by
`StaffRequiredMixin`), and contact PII is shown only for rows whose submitter opted in --
which none of the four demo runs did, so `contact_person`/`contact_email` render blank for
all of them below.

In [13]:
response = public_client.get(reverse('campaigns:table', args=[campaign.pk]))
assert response.status_code == 200

table = response.context['table']
print(f'Public table rows for {campaign.name!r} (campaign pk={campaign.pk}):')
for row in table.page.object_list:
    record = row.record
    print(
        f'  pk={record["pk"]}  telescope_instrument={record["telescope_instrument"]!r}  '
        f'approval_status={record["approval_status"]!r}  contact_person={record["contact_person"]!r}  '
        f'contact_email={record["contact_email"]!r}'
    )

visible_pks = {row.record['pk'] for row in table.page.object_list}
assert visible_pks == {classical_run.pk, lco_queue_run.pk, eso_queue_run.pk, class_wide_run.pk}

Public table rows for 'Campaign Lifecycle Demo' (campaign pk=1):
  pk=27  telescope_instrument='UT1/FORS2'  approval_status='approved'  contact_person=''  contact_email=''
  pk=26  telescope_instrument='Y22 1m0-SciCam-Sinistro'  approval_status='approved'  contact_person=''  contact_email=''
  pk=25  telescope_instrument='Y21 0.4m/SBIG-STX16803'  approval_status='approved'  contact_person=''  contact_email=''
  pk=28  telescope_instrument='LCO 1m0 Network (Campaign Lifecycle Demo)'  approval_status='approved'  contact_person=''  contact_email=''


## Summary

Across this notebook, one campaign and four `CampaignRun`s went from public submission
(`campaigns:submit`, anonymous) to fully on the calendar: staff approved all four through
the approval queue (`campaigns:decide` with `action=approve`); the one run whose free-text
site didn't resolve (the ESO run) surfaced in Sites Needing Review and was resolved
through a second, explicit staff action (`action=resolve_site`); one pre-existing,
hand-entered calendar event was attributed to the classical run through the attribution
queue (`campaigns:attribution_decide` with `action=confirm`); and the reconciler -- invoked
automatically by every one of those staff actions, never run as a separate batch command
here -- projected all four runs' `CalendarEvent` rows. Three of the four runs (classical,
LCO queue, ESO queue) ended up with one date-bearing `RUN:{pk}:{date}` event per observing
night; only the class-wide run, carrying a `telescope_class`, got a single bare `RUN:{pk}`
container spanning its whole window -- a run's `source` field never decides that shape.

See `docs/runbooks/telescope_runs_calendar.rst`, "How do I get every campaign run onto the
calendar?", for the operator-facing version of the window-shape rule demonstrated above,
and `reconcile_campaign_runs_demo.ipynb` for the reconciler command itself (`--dry-run`
counters, idempotency, sweeps and backfills) -- deliberately not duplicated here.